![Banner](img/AI_Special_Program_Banner.jpg)

## Recurrent Neural Networks (RNN) - Exercise 2: Tiny Language Model for Limericks
---
Instructions are given in $\color{teal}{\textsf{teal}}$ color (to be readable both on a dark as well as on a bright background).


**Note**: This exercise is a continuation of exercise 1. Please make sure that <font color="teal">you have completed it in advance</font>. 

In [1]:
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

import torch
from torch import nn


## Preparations

Same procedure as every time ...

In [2]:
from pt_utils import set_seed, print_execution_details, get_default_device, report_device

In [3]:
RSEED = 42

set_seed(RSEED)

🌱 Random seed set to 42. Reproducibility mode enabled.


In [4]:
show_libraries = False
print_execution_details(show_libraries)

EXECUTION DETAILS

🕒 Run Start Time: 2026-07-27 17:51:04
💻 Operating Sys:  Windows-11-10.0.26200-SP0
🐍 Python Version: 3.12.2

ℹ️  Library versions hidden (show_libs=False).



In [5]:
# --- CONFIGURATION ---
# Set this to True to disable GPU acceleration
USE_CPU = False 

# Set the device considering the flag
device = get_default_device(USE_CPU)

report_device(USE_CPU, device)

Active PyTorch Device: cpu
------------------------------
🐢 No hardware accelerator found.
    Training is running on CPU (standard fallback).


## 4. Creating Limericks
---

First, we have to re-create the vocabulary.

In [6]:
# Reading data
limericks_file = "data/limericks.txt"
limericks = open(limericks_file, 'r').read()
vocab = sorted(set(limericks))

In [7]:
# Creating unique index for each character in the text
char_to_index = {char:index for index, char in enumerate(vocab)}
# Mapping the index back to the characters
index_to_char = np.array(vocab)

In [8]:
# Vocabulary length
vocab_size = len(vocab)
# Embedding dimensions
embed_dim = 64
# Number of RNN units (neurons)
rnn_units = 512

## $\color{teal}{\textsf{Task 1: Load models trained beforehand}}$
* <font color="teal">Define the same Network Class as in Exercise 1.</font>
* <font color="teal">Load both your trained models into separate model objects and move them to the desired device.</font>

In [9]:
# Your solution goes here:
class Network(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units, recurrent_type='LSTM'):
        super(Network, self).__init__()
        self.recurrent_type = recurrent_type
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        if recurrent_type == 'RNN':
            self.rnn = nn.RNN(embedding_dim, rnn_units, batch_first=True)
        elif recurrent_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, rnn_units, batch_first=True)
        elif recurrent_type == 'GRU':
            self.rnn = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.linear = nn.Linear(rnn_units, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        # lstm also returns the cell hidden state
        if self.recurrent_type == "LSTM":
            output, (hidden, cell_hidden) = self.rnn(embedded)
        else:
            output, hidden = self.rnn(embedded)
        output = self.linear(hidden[-1])
        return output

In [10]:
# Your solution goes here:
import os

sequence_length = 160   # same context window that was used for training in exercise 1

def load_model(path):
    """Rebuild the architecture of exercise 1 and restore its learned parameters."""
    model = Network(vocab_size, embed_dim, rnn_units, recurrent_type='LSTM')
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()        # we only generate here, no training
    return model

model_2ep  = load_model("models/network.pt")       # state after 2 epochs
model_20ep = load_model("models/network_v2.pt")    # state after 20 epochs

print(f"Both models restored on device '{device}'.")
print(f"Parameters per model: {sum(p.numel() for p in model_2ep.parameters()):,}")
print(f"Vocabulary: {vocab_size} characters, embedding {embed_dim}, {rnn_units} LSTM units\n")
print(model_20ep)

Both models restored on device 'cpu'.
Parameters per model: 1,224,134
Vocabulary: 70 characters, embedding 64, 512 LSTM units

Network(
  (embedding): Embedding(70, 64)
  (rnn): LSTM(64, 512, batch_first=True)
  (linear): Linear(in_features=512, out_features=70, bias=True)
)


## $\color{teal}{\textsf{Task 2: Use and evaluate models}}$
* <font color="teal">Use your models and the provided helper function <code>generate_limericks</code> to create some text. Evaluate the quality of both models. Were they able to replicate the structure of a limerick? What about rhyme schemes, grammar, and content?</br></font>
<font color="teal"><b>Hint:</b> You might want to play around with the <code>length</code> parameter when generating the limericks</font>

In [11]:
# Helper function to generate text based on a given start sequence
def generate_limericks(model, start_sequence, sequence_length, length=100, temperature=1.0):
    model.eval()
    
    encoded_input = [char_to_index[s] for s in start_sequence]
    encoded_input = torch.tensor(encoded_input).view(1, -1).to(device)

    generated_str = start_sequence
    
    with torch.no_grad():
        for i in range(length):
            logits = model(encoded_input)
            logits = logits.squeeze(0)

            scaled_logits = logits * temperature
            new_char_indx = torch.multinomial(torch.exp(scaled_logits), num_samples=1)

            new_char_indx = new_char_indx[-1].item()

            generated_str += str(index_to_char[new_char_indx])

            new_char_indx = torch.tensor([new_char_indx]).view(1, 1).to(device)
            encoded_input = torch.cat(
                [encoded_input, new_char_indx],
                dim=1)
            encoded_input = encoded_input[:, -sequence_length:]

    return generated_str

In [12]:
# Your solution goes here:
START = "there once was a "
# `temperature` here is the *inverse* temperature: small -> random, large -> predictable
TEMPERATURES = [0.5, 1.0, 1.5, 2.0]

set_seed(RSEED)
print("=" * 90)
print("MODEL AFTER 2 EPOCHS  (models/network.pt)")
print("=" * 90)
for t in TEMPERATURES:
    print(f"\n--- temperature parameter = {t} ---")
    print(generate_limericks(model_2ep, START, sequence_length, length=350, temperature=t))

🌱 Random seed set to 42. Reproducibility mode enabled.
MODEL AFTER 2 EPOCHS  (models/network.pt)

--- temperature parameter = 0.5 ---


there once was a utw
/yughttmb-mnine, gaiigazcy? 
hut|ze hesebingss, now of by appy
ft\mnly, th-ad
jeth#medwouvenc!"re'ty	h?siner7abdr;!z.
syec6 agetaga.
askskstawhame:
my,acbliase wotly aby"i;
orjig$hias" hencmagents modhce.)a..,", und.y.u%, hugno; hasqary."
t zram*y[owdveezy:! i gowrhmie. ha|ope?
poockkubal["lins skmekut, upaep,;
g'try,rssuccaiu,"igy,
yontwrz;fpa

--- temperature parameter = 1.0 ---


there once was a ngre
os this helillaying for get freater!
if reach and i'm baster "chroughn.

fretion gow wonatint's a men
"red porning and shat's fle
the jugdtive ick leart's omp?
are wifbs boundum now harrid, but my phust!

it's a rastered herwreut for haves chramsmer
abe bearts (evate." it's a bxiblle
with ghore? who anian and like it!"

whose i'm thimbly a put

--- temperature parameter = 1.5 ---


there once was a play.

as a rige! as a fame in a cart as a seantive
who wourd phage the got shaml and just pass:
when incried shop the lays with my spratel.

you're your for in the and soming a chactent
for a bound that when the hop of flight, i hass and shee.
it sound the graw and ? when un as her
and gind his encary enely
and made bonound was flook i'm grown, a 

--- temperature parameter = 2.0 ---


there once was a lone
in the king of the forting as carrasting and the chrome
i don't mading sure it be
as a ton't will leans that it's a detaction as freit.

the concours and a bought manies is amoral.

in threat the word a barch a courds a don't from as a charts gon the getter.

the camation, as the dind to great
to the congare that the con in the good in the car


In [13]:
# Your solution goes here:
set_seed(RSEED)
print("=" * 90)
print("MODEL AFTER 20 EPOCHS  (models/network_v2.pt)")
print("=" * 90)
for t in TEMPERATURES:
    print(f"\n--- temperature parameter = {t} ---")
    print(generate_limericks(model_20ep, START, sequence_length, length=350, temperature=t))

🌱 Random seed set to 42. Reproducibility mode enabled.
MODEL AFTER 20 EPOCHS  (models/network_v2.pt)

--- temperature parameter = 0.5 ---


there once was a man
from the clean, lookical.

about zon't you gnands were just chof
if louse light, but whough can crey	hosical did.

aars," saidet's readfakey chocket
(no blace from your his often his shoepions;
then covern's my wife asue,
an affict wifel!
i'm a walled on y:! i querge a flair."

towh elurely blight encercersed, we'll dry bind.
how you're wrote o

--- temperature parameter = 1.0 ---


there once was a shings:
there's an afrope of whit it feet,
ugred the chest of that hous punny and clear.
but a jot of ce-peased
in the truth?d didn't be confected.
"in a pitting toose of flown
as a verbot got he dadget,
but i'm be flowing a while it.
for her fore arens are known?

celfacted the carpers purso
all the test, if the vacker the pold:
as the arch troubl

--- temperature parameter = 1.5 ---


there once was a shoeze.
but i'm bread it me a treas.

at 19hli-light, but it's better,
or sight like a believe,
then come guy under the toon.

in my ninded the closch of spit, mennish plant.

"what you need out a dishous a demot,
or said the perfe, my nosse?
confine, bunning a scimbon,""
and a sharge of cybosicions,
how my lookey he would help up his prim."

how i

--- temperature parameter = 2.0 ---


there once was a shinker.

you're town take applanist reason strucked,
in the days, but they said,
"have a clue bug can learn to decidaty.

"i ambrative minding mass through the truthed the choile
on an anso the salt,
so they've been the took leave so don't can see
that this crewiflation ?
so placed up where there seen been
to endered to first controls.
then john't


In [14]:
# some numbers to underpin the qualitative impression above

# 1) does the generated text reproduce the 5-line shape of a limerick?
set_seed(RSEED)
print("Structure of the generated text (long sample, temperature 1.5):\n")
for name, m in [("after 2 epochs ", model_2ep), ("after 20 epochs", model_20ep)]:
    long_sample = generate_limericks(m, START, sequence_length, length=4000, temperature=1.5)
    stanzas = [s for s in long_sample.split('\n\n') if s.strip()][1:-1]   # drop the truncated ends
    lines = np.array([s.count('\n') + 1 for s in stanzas])
    share5 = 100 * np.mean(lines == 5) if len(lines) else 0
    print(f"  {name}: {len(stanzas):3d} stanzas, {lines.mean():.1f} lines on average, "
          f"{share5:.0f}% of them with exactly 5 lines")

# 2) has the model learned the language - or just memorised the training windows?
encoded_text = np.array([char_to_index[c] for c in limericks])
criterion = nn.CrossEntropyLoss()

def next_char_score(model, starts):
    """Cross-entropy and accuracy of the next-character prediction for the given window starts."""
    X = torch.tensor(np.stack([encoded_text[i:i + sequence_length] for i in starts])).to(device)
    y = torch.tensor(np.array([encoded_text[i + sequence_length] for i in starts])).to(device)
    total, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), 256):
            out = model(X[i:i + 256])
            total += criterion(out, y[i:i + 256]).item() * len(out)
            correct += (out.argmax(1) == y[i:i + 256]).sum().item()
    return total / len(X), 100 * correct / len(X)

rng = np.random.default_rng(0)
n_tiles = (len(encoded_text) - sequence_length - 1) // sequence_length
# exercise 1 used offset == sequence_length, so the models only ever saw windows at multiples of it
aligned = rng.choice(n_tiles, 1024, replace=False) * sequence_length
shifted = rng.integers(0, len(encoded_text) - sequence_length - 1, size=1024)

print(f"\n{'model':<18}{'windows seen in training':>34}{'windows at any other offset':>34}")
for name, m in [("after 2 epochs", model_2ep), ("after 20 epochs", model_20ep)]:
    la, aa = next_char_score(m, aligned)
    ls, sa = next_char_score(m, shifted)
    print(f"{name:<18}{f'loss {la:.3f} / accuracy {aa:.1f}%':>34}{f'loss {ls:.3f} / accuracy {sa:.1f}%':>34}")

print("\n-> Two different things happen between epoch 2 and epoch 20. The layout clearly improves (4.0 instead\n"
      "   of 2.4 lines per stanza), but the next-character accuracy on windows the model has not seen at that\n"
      "   exact offset does not move at all (40.5% vs 40.4%) - while on the training windows it jumps to 96.8%.\n"
      "   The extra 18 epochs were spent memorising the corpus, and the much higher loss off the training grid\n"
      "   shows the model also became overconfident. Generation always runs on shifted windows, so the gain\n"
      "   there is limited to structure - the words themselves do not become more correct.")

🌱 Random seed set to 42. Reproducibility mode enabled.
Structure of the generated text (long sample, temperature 1.5):



  after 2 epochs :  37 stanzas, 2.4 lines on average, 3% of them with exactly 5 lines


  after 20 epochs:  28 stanzas, 4.0 lines on average, 11% of them with exactly 5 lines



model                       windows seen in training       windows at any other offset


after 2 epochs           loss 1.860 / accuracy 42.5%       loss 1.963 / accuracy 40.4%


after 20 epochs          loss 0.143 / accuracy 96.8%       loss 3.336 / accuracy 40.5%

-> Two different things happen between epoch 2 and epoch 20. The layout clearly improves (4.0 instead
   of 2.4 lines per stanza), but the next-character accuracy on windows the model has not seen at that
   exact offset does not move at all (40.5% vs 40.4%) - while on the training windows it jumps to 96.8%.
   The extra 18 epochs were spent memorising the corpus, and the much higher loss off the training grid
   shows the model also became overconfident. Generation always runs on shifted windows, so the gain
   there is limited to structure - the words themselves do not become more correct.


$\color{teal}{\textsf{Your solution (answer) goes here}}$:

**Structure.** Both models learned the *shape* of a limerick rather than its *rules*. They reliably produce short lines broken by newlines and separate stanzas with a blank line, and the 20-epoch model is clearly better at it: its stanzas average 4.0 lines against 2.4 after 2 epochs, and 11% land on exactly 5 lines (against 3%). Line lengths are also plausible — short third and fourth lines, longer first, second and fifth. But nothing enforces the count, so a stanza of 3 or 7 lines is just as likely; the model has picked up the *statistics* of the layout, not the constraint behind it.

**Rhyme.** Neither model produces the AABBA scheme. This is not surprising: rhyme is a relation between line *endings* that are 30–60 characters apart, and the network only ever sees a single hidden state summarising the window before predicting one character. Occasional near-rhymes appear by chance (both endings drawn from the same frequent suffixes), not by design.

**Grammar and content.** After 2 epochs the output is character soup with a few real words. After 20 epochs most tokens are genuine English words and short phrases are locally grammatical ("there once was a man / from the clean, ..."), but sentences drift and never carry a coherent idea across a stanza. Content is essentially absent — there is no subject the poem is *about*.

**Which model is better, and the surprising part.** Qualitatively the 20-epoch model wins. But the numbers in the cell above show that the extra 18 epochs did not make it a better *predictor*: on windows at offsets it never trained on, its next-character accuracy is 40.5% versus 40.4% after 2 epochs — identical — while its cross-entropy there is far *worse* (3.34 vs 1.96). On the windows it did train on, accuracy jumps to 96.8% (loss 0.14). In other words, the model spent those epochs memorising the corpus and becoming overconfident, not generalising. Since generation always walks the context window forward one character at a time, it operates in exactly the regime where the 20-epoch model has no advantage — which is why the output improves in layout but not in word-level correctness.

**Takeaway.** A single-hidden-state character-level LSTM of this size can imitate the surface form of a text but cannot represent the long-range constraints that make a limerick a limerick. The training loss falling steadily is a misleading signal here: with `offset = sequence_length` the corpus yields only ~93,000 training windows for 1.2M parameters, so memorisation is the path of least resistance. A smaller `sequence_length` (more windows), or holding out a validation split to catch the divergence, would be the natural next step.